# ECE 505 – Hardware Security: Project 2 (c7552 SAT Solution Finder)

This notebook loads the `c7552` CNF (`5,053` base clauses) and the `875` wire constraints (`583` positives + `292` negatives = `5,928` total clauses), benchmarks all 10 PySAT solvers, and exports `SAT.xlsx`.

In [ ]:
# 1. Clone GitHub repository (replace URL with your repo) and install dependencies
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"
REPO_DIR = "ece505_project2_github"

import os
if not os.path.exists(REPO_DIR) and "<your-username>" not in REPO_URL:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
elif os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull

!pip install -q -r requirements.txt

In [ ]:
# 2. Validate CNF and Wire Constraints
import sys
sys.path.insert(0, "src")

from find_solutions import (
    SOLVER_MAP,
    benchmark_solvers,
    build_constraint_clauses,
    find_first_n_solutions,
    load_cnf,
    load_constraints_from_json,
    load_node_groups,
)

base_clauses = load_cnf("data/cnf.txt")
positives, negatives = load_constraints_from_json("data/constraints.json")
constraint_clauses = build_constraint_clauses(positives, negatives, "data/node_index.txt")
all_clauses = base_clauses + constraint_clauses
groups = load_node_groups("data/node_index.txt")

print(f"Base clauses: {len(base_clauses)}")
print(f"Constraint clauses: {len(constraint_clauses)} ({len(positives)} positives, {len(negatives)} negatives)")
print(f"Total clauses: {len(all_clauses)}")
print("Node groups:", {k: len(v) for k, v in groups.items()})

In [ ]:
# 3. Run all 10 PySAT Solvers & Save SAT.xlsx
import json
import pandas as pd

PROJECTION_MODE = "all"  # options: "all", "inputs", "outputs"
target_vars = (
    groups["INPUT"] if PROJECTION_MODE == "inputs"
    else groups["OUTPUT"] if PROJECTION_MODE == "outputs"
    else None
)

MAX_SOLUTIONS = 100   # set to None for unlimited until UNSAT or timeout
TIMEOUT_S = 30.0      # per-solver timeout in seconds

rows, raw_details = benchmark_solvers(
    clauses=all_clauses,
    solver_names=list(SOLVER_MAP.keys()),
    n=MAX_SOLUTIONS,
    target_vars=target_vars,
    timeout_s=TIMEOUT_S,
)

df = pd.DataFrame(rows)
display(df)

with pd.ExcelWriter("SAT.xlsx", engine="openpyxl", mode="w") as writer:
    df.to_excel(writer, sheet_name="Sheet1", index=False)

with open("benchmark_raw.json", "w", encoding="utf-8") as f:
    json.dump({"summary": rows, "solvers": raw_details}, f, indent=2)

print("Saved SAT.xlsx and benchmark_raw.json successfully.")

In [ ]:
# 4. Inspect Sample Satisfying Solutions (Minisat22)
sample_sols, elapsed, status = find_first_n_solutions(
    solver_cls=SOLVER_MAP["Minisat22"],
    clauses=all_clauses,
    n=5,
    target_vars=target_vars,
    timeout_s=15.0,
)

print(f"Status: {status} | Elapsed: {elapsed:.6f}s | Found: {len(sample_sols)}")
for idx, sol in enumerate(sample_sols, 1):
    print(f"Solution {idx} (first 25 vars): {sol[:25]}")